# Implementation of Enhanced RED CNN - with MAR dataset
Inspired by our forked copy from SSinyu/RED-CNN, edited by Nili and Shoham
(insert IN THE END picture, too annoying)

Running Order:
1.   Cell 4 (prep.py)
2.   Cell 7 (main.py) - for train
3.   Cell 7 (main.py) - for test



**Cell** **0**: Improts  and Installing

In [ ]:
!pip install pydicom
!pip install optuna
import random
import os
import time
import argparse
from glob import glob
from collections import OrderedDict
from math import exp


import numpy as np
import pydicom
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Variable
from torch.backends import cudnn
import optuna
import scipy.ndimage as ndimage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 31.1 MB/s eta 0:00:00


**Cell 1:** upload data from drive and organize it properly

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive/3mmB30')

#!gdown --folder https://drive.google.com/drive/folders/1cvmktRMZHnKOnpT9U77i_aZ56xMsNQ59?usp=sharing

# Download the single zip file using its ID
# !gdown --id 1Cy8fxEx0mgNhiBv4slc_pF8YoqMICb6K -O DATA_MAR.zip
!gdown --id 1uuXczaaPQxBfhE4iPiZQdY8_R1B1mVNk -O DATA_MAR.zip
# Unzip the contents into your Colab environment and remove the zip
!unzip -q DATA_MAR.zip -d /content/DATA_MAR
!rm DATA_MAR.zip

!mv /content/DATA_MAR/*MAR*_*dataset*/t* /content/DATA_MAR/
!mv /content/DATA_MAR/*MAR*_*dataset*/v* /content/DATA_MAR/
!rm -r /content/DATA_MAR/*MAR*_*dataset*

#fix some transparent characters:
validation_path = '/content/DATA_MAR/validation'

if os.path.exists(validation_path):
    for item in os.listdir(validation_path):
        # Create a clean version of the string keeping only letters and numbers
        clean_name = "".join(c for c in item if c.isalnum())

        old_path = os.path.join(validation_path, item)
        new_path = os.path.join(validation_path, clean_name)

        # If the name contained hidden characters, rename the folder
        if old_path != new_path:
            os.rename(old_path, new_path)
            print(f"Fixed: {repr(item)} is now {clean_name}")

    print("Cleanup complete. Your normal paths will work now.")
else:
    print(f"Directory not found: {validation_path}")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1uuXczaaPQxBfhE4iPiZQdY8_R1B1mVNk
From (redirected): https://drive.google.com/uc?id=1uuXczaaPQxBfhE4iPiZQdY8_R1B1mVNk&confirm=t&uuid=a22c7782-b6d8-464f-b821-55a01028ee74
To: /content/DATA_MAR.zip
100% 4.33G/4.33G [02:26<00:00, 29.6MB/s]
Fixed: '\u200f\u200fbody8' is now body8
Fixed: '\u200f\u200fbody13' is now body13
Cleanup complete. Your normal paths will work now.


**Cell 2:** Model class description, including all layers and feed forward method.
*   **Encoder**: first 5 conv layers
*   **Decoder**: last 5 deconv layers, including skip connections.

Every Conv/Deconv layer, starting from the second, works on 96 feature channels in a 5x5 kernel, as decided emprically by the original creatores of RED CNN, since it captures enough details without making the model too heavy.

**Based on:** networks.py

In [ ]:
class RED_CNN_Advanced(nn.Module):
    """
    Advanced RED-CNN Model for Metal Artifact Reduction (MAR).
    Upgrades include:
    1. Dilated convolutions in the encoder to increase the receptive field without adding parameters.
    2. Instance Normalization for better style/artifact decoupling.
    3. U-Net style concatenation skip-connections instead of simple addition.

    Notation for dimensions: (B, C, H, W)
    B = Batch Size
    C = Channels
    H = Height of image/patch
    W = Width of image/patch
    """
    def __init__(self, out_ch=96):
        super(RED_CNN_Advanced, self).__init__()

        # ENCODER
        # Standard convolution to extract initial features.
        # Padding=2 ensures the spatial size remains H x W.
        self.conv1 = nn.Conv2d(1, out_ch, kernel_size=5, stride=1, padding=2)
        self.in1 = nn.InstanceNorm2d(out_ch)

        # Dilated convolutions (dilation=2).
        # To keep spatial dimensions constant, padding = dilation * (kernel-1) // 2
        # padding = 2 * (5-1) // 2 = 4
        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=5, stride=1, padding=4, dilation=2)
        self.in2 = nn.InstanceNorm2d(out_ch)

        self.conv3 = nn.Conv2d(out_ch, out_ch, kernel_size=5, stride=1, padding=4, dilation=2)
        self.in3 = nn.InstanceNorm2d(out_ch)

        self.conv4 = nn.Conv2d(out_ch, out_ch, kernel_size=5, stride=1, padding=4, dilation=2)
        self.in4 = nn.InstanceNorm2d(out_ch)

        self.conv5 = nn.Conv2d(out_ch, out_ch, kernel_size=5, stride=1, padding=4, dilation=2)
        self.in5 = nn.InstanceNorm2d(out_ch)

        # DECODER
        # Standard Transposed Convolutions. Padding=2 maintains spatial size.
        self.tconv1 = nn.ConvTranspose2d(out_ch, out_ch, kernel_size=5, stride=1, padding=2)
        self.in_t1 = nn.InstanceNorm2d(out_ch)

        # This layer receives a U-Net style concatenated input.
        # It takes the output of tconv1 (out_ch) + the skip connection (out_ch) = out_ch * 2
        self.tconv2 = nn.ConvTranspose2d(out_ch * 2, out_ch, kernel_size=5, stride=1, padding=2)
        self.in_t2 = nn.InstanceNorm2d(out_ch)

        self.tconv3 = nn.ConvTranspose2d(out_ch, out_ch, kernel_size=5, stride=1, padding=2)
        self.in_t3 = nn.InstanceNorm2d(out_ch)

        # This layer also receives a U-Net style concatenated input (out_ch * 2).
        self.tconv4 = nn.ConvTranspose2d(out_ch * 2, out_ch, kernel_size=5, stride=1, padding=2)
        self.in_t4 = nn.InstanceNorm2d(out_ch)

        # Final layer reduces channels back to 1 (grayscale image).
        self.tconv5 = nn.ConvTranspose2d(out_ch, 1, kernel_size=5, stride=1, padding=2)

        # Activation function
        self.relu = nn.ReLU()

    def forward(self, x):
        """
        Feed-forward pass tracking tensor dimensions.
        Input `x` shape: (B, 1, H, W)
        """
        # Save the global input for the final residual connection
        residual_1 = x  # Shape: (B, 1, H, W)

        # Encoder Pass
        # Conv1: Extracts 96 feature maps
        out1 = self.relu(self.in1(self.conv1(x)))  # Shape: (B, 96, H, W)

        # Conv2: Dilated convolution
        out2 = self.relu(self.in2(self.conv2(out1)))  # Shape: (B, 96, H, W)
        residual_2 = out2  # Save for deep U-Net skip connection later. Shape: (B, 96, H, W)

        # Conv3 & Conv4: Further dilated feature extraction
        out3 = self.relu(self.in3(self.conv3(out2)))  # Shape: (B, 96, H, W)
        out4 = self.relu(self.in4(self.conv4(out3)))  # Shape: (B, 96, H, W)
        residual_3 = out4  # Save for early U-Net skip connection later. Shape: (B, 96, H, W)

        # Conv5: Bottleneck of the encoder
        out5 = self.relu(self.in5(self.conv5(out4)))  # Shape: (B, 96, H, W)

        # Decoder Pass
        # TConv1: Begin decoding
        dout1 = self.relu(self.in_t1(self.tconv1(out5)))  # Shape: (B, 96, H, W)

        # Early U-Net Skip Connection: Concatenate tconv1 output with conv4 output across the channel dimension (dim=1)
        dout1_concat = torch.cat([dout1, residual_3], dim=1)  # Shape: (B, 192, H, W)

        # TConv2: Process the concatenated features, reducing channels back to 96
        dout2 = self.relu(self.in_t2(self.tconv2(dout1_concat)))  # Shape: (B, 96, H, W)

        # TConv3: Continue decoding
        dout3 = self.relu(self.in_t3(self.tconv3(dout2)))  # Shape: (B, 96, H, W)

        # Deep U-Net Skip Connection: Concatenate tconv3 output with conv2 output
        dout3_concat = torch.cat([dout3, residual_2], dim=1)  # Shape: (B, 192, H, W)

        # TConv4: Process concatenated features
        dout4 = self.relu(self.in_t4(self.tconv4(dout3_concat)))  # Shape: (B, 96, H, W)

        # TConv5: Final projection to single channel
        out = self.tconv5(dout4)  # Shape: (B, 1, H, W)

        # Global Residual Connection: Add the original input to the generated artifact-free prediction
        out += residual_1  # Shape: (B, 1, H, W) + (B, 1, H, W) = (B, 1, H, W)

        return self.relu(out)  # Final Output Shape: (B, 1, H, W)

**Cell 3:** Metrics for Evaluting the net (image) output. For metal artifacts removal, we aplly the merics only at Roi (the regions around metal objects, which contain the artifacts).
The cell contain primary function "compute_measure" that evaluting the output using different methods, such as:

1.   **PSNR**: Measure Peak Signal to Noise Ratio. Peak Signal is (data_range)^2. data_range = trunc_max - trunc_min. May be different between Mayo images and MAR images. Noise is the average power of the error signal. Higher PSNR expected from the net output.
2.   **SSIM**: Measure the average Structural Similarity Index (composed of 3 comparison kriteria: luminance, contrast, structure). Higher SSIM expected from the net output.
3.   **RMSE**: Measure the Root Mean Square Error between GT and input/output images. Less error expected from the net output.

**Based on:** measure.py

In [ ]:
# ----
# Mask Generation & Main Evaluating func (Masked)
# ----
def get_roi_masks(mask_path, shape=(512, 512), dilation_iterations=20):
    mask_data = np.fromfile(mask_path, dtype=np.float32).reshape(shape)
    metal_mask = mask_data > 0.5
    dilated_mask = ndimage.binary_dilation(metal_mask, iterations=dilation_iterations)
    artifact_mask = dilated_mask ^ metal_mask
    background_mask = ~dilated_mask
    return metal_mask, artifact_mask, background_mask

def compute_measure(x, y, pred, data_range, mask=None):
    original_psnr = compute_PSNR(x, y, data_range, mask)
    original_ssim = compute_SSIM(x, y, data_range, mask=mask)
    original_rmse = compute_RMSE(x, y, mask)
    pred_psnr = compute_PSNR(pred, y, data_range, mask)
    pred_ssim = compute_SSIM(pred, y, data_range, mask=mask)
    pred_rmse = compute_RMSE(pred, y, mask)
    return (original_psnr, original_ssim, original_rmse), (pred_psnr, pred_ssim, pred_rmse)
# ----
# 1. RMSE & MSE (Masked)
# ----
def compute_MSE(img1, img2, mask=None):
    if mask is not None:
        # error computed only in roi pixels
        return ((img1[mask] - img2[mask]) ** 2).mean()
    return ((img1 - img2) ** 2).mean()

def compute_RMSE(img1, img2, mask=None):
    mse_ = compute_MSE(img1, img2, mask)
    if type(img1) == torch.Tensor:
        return torch.sqrt(mse_).item()
    else:
        return np.sqrt(mse_)

# ----
# 2. PSNR (Masked)
# ----
def compute_PSNR(img1, img2, data_range, mask=None):
    mse_ = compute_MSE(img1, img2, mask)
    if mse_ == 0: # avoid 0 sub
        return float('inf')

    if type(img1) == torch.Tensor:
        return 10 * torch.log10((data_range ** 2) / mse_).item()
    else:
        return 10 * np.log10((data_range ** 2) / mse_)

# ----
# 3. SSIM (Masked)
# ----
def compute_SSIM(img1, img2, data_range, window_size=11, channel=1, size_average=True, mask=None):
    if len(img1.size()) == 2:
        shape_ = img1.shape[-1]
        img1 = img1.view(1,1,shape_, shape_)
        img2 = img2.view(1,1,shape_, shape_)

    window = create_window(window_size, channel)
    window = window.type_as(img1)

    mu1 = F.conv2d(img1, window, padding=window_size//2)
    mu2 = F.conv2d(img2, window, padding=window_size//2)
    mu1_sq, mu2_sq = mu1.pow(2), mu2.pow(2)
    mu1_mu2 = mu1*mu2

    sigma1_sq = F.conv2d(img1*img1, window, padding=window_size//2) - mu1_sq
    sigma2_sq = F.conv2d(img2*img2, window, padding=window_size//2) - mu2_sq
    sigma12 = F.conv2d(img1*img2, window, padding=window_size//2) - mu1_mu2

    C1, C2 = (0.01*data_range)**2, (0.03*data_range)**2

    # SSIM of entire image:
    ssim_map = ((2*mu1_mu2+C1)*(2*sigma12+C2)) / ((mu1_sq+mu2_sq+C1)*(sigma1_sq+sigma2_sq+C2))

    if mask is not None:
        # fitting mask for tensor format, extract mean only in roi
        mask_tensor = torch.from_numpy(mask).to(ssim_map.device).view(1, 1, shape_, shape_)
        return ssim_map[mask_tensor].mean().item()

    if size_average:
        return ssim_map.mean().item()
    else:
        return ssim_map.mean(1).mean(1).mean(1).item()

def gaussian(window_size, sigma):
    gauss = torch.Tensor([exp(-(x - window_size // 2) ** 2 / float(2 * sigma ** 2)) for x in range(window_size)])
    return gauss / gauss.sum()

def create_window(window_size, channel):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = Variable(_2D_window.expand(channel, 1, window_size, window_size).contiguous())
    return window


class CombinedLoss(nn.Module):
    """
    A custom loss function combining L1 (Mean Absolute Error) and SSIM (Structural Similarity).
    L1 is less sensitive to outliers than MSE, keeping edges sharp.
    SSIM ensures the overall structural integrity of the tissue is preserved.
    """
    def __init__(self, alpha=0.84, data_range=1.0):
        super(CombinedLoss, self).__init__()
        self.alpha = alpha # Alpha controls the balance. 0.84 is historically optimal for SSIM+L1. ADD REFRENCE IN REPORT!!!!
        self.l1_loss = nn.L1Loss()
        self.data_range = data_range

    def forward(self, pred, target):
        # Calculate standard L1 distance between prediction and ground truth
        l1 = self.l1_loss(pred, target)

        # Calculate SSIM using the function provided in your existing `measure.py`
        ssim_val = compute_SSIM(pred, target, data_range=self.data_range, size_average=True)

        # SSIM outputs a score from -1 to 1, where 1 is a perfect match.
        # Since optimizers minimize loss, we subtract the SSIM score from 1.
        ssim_loss = 1 - ssim_val

        # Return the weighted combination
        return (self.alpha * ssim_loss) + ((1 - self.alpha) * l1)

**Cell 4:** Pre-Processing of the data. Process and organize the images:
*   Normalize the data
*   Convert all RAW images to numpy format

**Based on**: prep.py

In [ ]:
def save_dataset(args):
    # Check if the directory specified for saving the processed numpy arrays exists
    if not os.path.exists(args.save_path):
        os.makedirs(args.save_path)
        print('Create path : {}'.format(args.save_path))
    # Iterate through the three main operational modes: 'train' 'test' 'validation' folders
    for mode in ['train', 'test', 'validation']:
        mode_path = os.path.join(args.data_path, mode)
        # Check if this specific mode directory actually exists on the disk
        if not os.path.exists(mode_path):
            print(f"Warning: Directory {mode_path} not found.")
            continue

        print(f"\nProcessing {mode} data...")
        # Define a list of the subfolders expected within the dataset
        body_folders = ['body5', 'body8', 'body13']

        for body in body_folders:
            corrupted_path = os.path.join(mode_path, body, 'corrupted')
            gt_path = os.path.join(mode_path, body, 'gt')

            # Verify that both the corrupted and gt subdirectories exist for this body part
            if not os.path.exists(corrupted_path) or not os.path.exists(gt_path):
                print(f"Warning: Missing subdirectories for {body} in {mode} data. Expected corrupted path: {corrupted_path}, Expected GT path: {gt_path}")
                continue

            # Find all files ending with '.raw' in the corrupted directory and sort them alphabetically
            corrupted_files = sorted(glob(os.path.join(corrupted_path, '*.raw')))
            total_files = len(corrupted_files)

            if total_files == 0:
                continue

            # Iterate through each individual corrupted '.raw' file found
            for i, c_file in enumerate(corrupted_files):
                filename = os.path.basename(c_file)
                gt_filename = filename.replace('metalart', 'nometal')
                gt_file = os.path.join(gt_path, gt_filename)

                if not os.path.exists(gt_file):
                    print(f"Warning: File {gt_file} Target label not found.")
                    continue

                # Read and normalize
                input_img = read_raw_image(c_file)
                target_img = read_raw_image(gt_file)

                # Normalize the images pixels values to fall between 0 and 1 based on HU bounds
                input_norm = normalize_(input_img, args.norm_range_min, args.norm_range_max)
                target_norm = normalize_(target_img, args.norm_range_min, args.norm_range_max)

                # Extract ID (e.g., extracting '4001' from the string) and save
                img_id = filename.split('_img')[1].split('_')[0]
                np.save(os.path.join(args.save_path, f'{mode}_{body}_{img_id}_input.npy'), input_norm)
                np.save(os.path.join(args.save_path, f'{mode}_{body}_{img_id}_target.npy'), target_norm)

                # Trigger the progress bar update after each file is saved
                printProgressBar(i + 1, total_files, prefix=f"Saving {body} ({mode}):", suffix='Complete', length=30)

        print(f"Finished saving all {mode} datasets.")

def normalize_(image, MIN_B=-1024.0, MAX_B=3072.0):
    """Normalizes HU values to a 0-1 scale."""
    image = (image - MIN_B) / (MAX_B - MIN_B)
    return image

def read_raw_image(filepath, shape=(512, 512), dtype=np.float32):
    """
    Reads a raw binary file.
    """
    img = np.fromfile(filepath, dtype=np.float32)
    return img.reshape(shape)


def printProgressBar(iteration, total, prefix='', suffix='', decimals=1, length=100, fill=' '):
    # referred from https://gist.github.com/snakers4/91fa21b9dda9d055a02ecd23f24fbc3d
    percent = ("""{0:.""" + str(decimals) + """f}""").format(100 * (iteration / float(total)))
    filledLength = int(length * iteration // total)
    bar = fill * filledLength + '=' * (length - filledLength)
    print('\r%s |%s| %s%% %s' % (prefix, bar, percent, suffix), end='\r')
    if iteration == total:
        print()

# This block only runs if this script is executed directly
if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    # Define where the raw DICOM files are located
    parser.add_argument('--data_path', type=str, default='./content/DATA_MAR/')

    # Define where the processed 3D NumPy arrays (.npy files) should be saved
    parser.add_argument('--save_path', type=str, default='./npy_img/')

    # Target slice thickness in millimeters (mm) to standardize the 3D volume resolution - not in use in this implementation
    parser.add_argument('--mm', type=int, default=400)

    # Define the windowing/clipping range for the Hounsfield Units.
    # Neural networks prefer data within a tight, normalized range.
    # Any tissue density below -1024 HU (air) or above 3072 HU (dense bone/metal)
    # will be clipped to these boundaries before being normalized between 0 and 1.
    parser.add_argument('--norm_range_min', type=float, default=-1024.0)
    parser.add_argument('--norm_range_max', type=float, default=3072.0)

    # For terminal: Parse all the provided arguments and store them in the 'args' variable
    # args = parser.parse_args()

    ########
    # For colab:
    # Passing 'args=[]' forces argparse to parse an empty list
    # instead of reading Jupyter/Colab's hidden system arguments (which causes crashes).
    # This essentially loads all the 'default' values defined above into the 'args' variable.
    args = parser.parse_args(args=[])
    # Since we bypassed the command line, we manually overwrite any variables
    # to match our Google Colab environment setup.
    args.data_path = "/content/DATA_MAR"
    args.save_path = "/content/npy_img"
    ########

    # Trigger the main preprocessing pipeline using these settings
    save_dataset(args)

Create path : /content/npy_img

Processing train data...
Saving body5 (train): |                              | 100.0% Complete
Saving body8 (train): |                              | 100.0% Complete
Saving body13 (train): |                              | 100.0% Complete
Finished saving all train datasets.

Processing test data...
Saving body5 (test): |                              | 100.0% Complete
Saving body8 (test): |                              | 100.0% Complete
Saving body13 (test): |                              | 100.0% Complete
Finished saving all test datasets.

Processing validation data...
Saving body5 (validation): |                              | 100.0% Complete
Saving body8 (validation): |                              | 100.0% Complete
Saving body13 (validation): |                              | 100.0% Complete
Finished saving all validation datasets.


**Cell 5:** Definition of Dataset Class and DataLoader object
*   **ct_dataset**: Class that holds all the patches of all the images of all the chosen patients (after desired transform).
*   **data_loader**: Creating batches and shuffling the data.




**Based on**: loader.py

In [ ]:
class ct_dataset(Dataset):
    """The MAR data set composed of 3 sub-datasets of body CT images. According to current main,
    92% of them for train. For each image we take 10 patches.
    that's make the data set size around 23,740 image samples (pairs of corrupted and ground-truth) in test and train sets.
    we overloaded 3 basic method of the dataset class: init, len and getitem"""
    def __init__(self, mode, load_mode, saved_path, patch_n=None, patch_size=None, transform=None):
        assert mode in ['train', 'test', 'validation'], "mode is 'train' (regular or only decoder), 'test' or 'validation'"
        assert load_mode in [0,1], "load_mode is 0 (slow) or 1 (fast, if RAM can spare 10GB)"

        # Fetch files matching the requested mode (train or test)
        self.input_ = sorted(glob(os.path.join(saved_path, f'{mode}_*_input.npy')))
        self.target_ = sorted(glob(os.path.join(saved_path, f'{mode}_*_target.npy')))

        self.load_mode = load_mode
        self.patch_n = patch_n # how many patches to take from single image
        self.patch_size = patch_size # if none, patch is entire image
        self.transform = transform # pre-proccessing or augmentation function

        if load_mode == 1:
            self.input_ = [np.load(f) for f in self.input_]
            self.target_ = [np.load(f) for f in self.target_]
        else:
            self.input_ = self.input_
            self.target_ = self.target_


    def __len__(self): # size of data (num of pairs (input, target))
        return len(self.target_)

    def __getitem__(self, idx): # returns all n patches of certain pair + their id number (input image, gt image, sample id taken from their name)
        basename = os.path.basename(self.input_[idx])
        img_id = basename.split('_')[2]
        ####### TRY TO PRINT IF IT TRULY CATCHES THE NUMBER!
        # it should work because the format is: f'{mode}_{body}_{img_id}_input.npy', 3rd element is id!
        input_img, target_img = self.input_[idx], self.target_[idx]
        if self.load_mode == 0:
            input_img, target_img = np.load(input_img), np.load(target_img)

        if self.transform:
            input_img = self.transform(input_img)
            target_img = self.transform(target_img)

        if self.patch_size:
            input_patches, target_patches = get_patch(input_img, target_img, self.patch_n, self.patch_size)
            return (input_patches, target_patches, img_id)
        else:
            return (input_img, target_img, img_id)


def get_patch(full_input_img, full_target_img, patch_n, patch_size, augment=True):
    """
    Extracts random patches from the full CT slice to save memory and increase dataset variety.
    Now includes Data Augmentation (random flips) to prevent overfitting.
    """
    assert full_input_img.shape == full_target_img.shape
    patch_input_imgs = []
    patch_target_imgs = []

    # Get dimensions of the full CT slice (e.g., 512x512)
    h, w = full_input_img.shape
    new_h, new_w = patch_size, patch_size

    for _ in range(patch_n):
        # Pick a random starting coordinate for the top-left corner of the patch
        # Bounded so the patch doesn't fall off the edge of the image
        top = np.random.randint(0, h - new_h)# each patch must start in valid place (not too much close to the edge)
        left = np.random.randint(0, w - new_w)

        # Slice the NumPy arrays to extract the patch
        patch_input = full_input_img[top:top+new_h, left:left+new_w]
        patch_target = full_target_img[top:top+new_h, left:left+new_w]

        # Data Augmentation Block
        if augment:
            # 50% chance to flip the image vertically (upside down)
            if random.random() > 0.5:
                patch_input = np.flip(patch_input, axis=0)
                patch_target = np.flip(patch_target, axis=0)

            # 50% chance to flip the image horizontally (left to right)
            if random.random() > 0.5:
                patch_input = np.flip(patch_input, axis=1)
                patch_target = np.flip(patch_target, axis=1)

        # Use .copy() to ensure the flipped arrays are stored continuously in memory,
        # which prevents PyTorch stride errors later on.
        patch_input_imgs.append(patch_input.copy())
        patch_target_imgs.append(patch_target.copy())

    # Return as stacked numpy arrays
    return np.array(patch_input_imgs), np.array(patch_target_imgs)

def get_loader(mode='train', load_mode=0, saved_path=None, patch_n=None, patch_size=None, transform=None, batch_size=32, num_workers=6):
    # Map both fine-tuning modes to 'train' so it loads the correct .npy files
    dataset_mode = 'train' if mode in ['train_decoder', 'train_fine_tune'] else mode
    dataset_ = ct_dataset(dataset_mode, load_mode, saved_path, patch_n, patch_size, transform)
    # Ensure we shuffle the data during any training mode
    is_train = mode in ['train', 'train_decoder', 'train_fine_tune']
    # check if it is good - that we shuffle only in train so in test, we can get the right mask file!
    data_loader = DataLoader(dataset=dataset_, batch_size=batch_size, shuffle=is_train, num_workers=num_workers)
    return data_loader

def printProgressBar(iteration, total, prefix='', suffix='', decimals=1, length=100, fill=' '):
    percent = ("{0:." + str(decimals) + "f}").format(100 * (iteration / float(total)))
    filledLength = int(length * iteration // total)
    bar = fill * filledLength + '=' * (length - filledLength)
    print('\r%s |%s| %s%% %s' % (prefix, bar, percent, suffix), end='\r')
    if iteration == total:
        print()

**Cell 6:** Definition of Solver Class, manager of all processes:

*   Training
*   Testing
*   Data processing
*   Saving figs and models

**Based on**: solver.py

In [ ]:
class Solver(object):
    def __init__(self, args, data_loader, valid_loader=None): # NEW argument
        """
        The Solver class handles the training and testing loops for the RED-CNN model.
        It manages data loading, model weight saving/loading, learning rate decay,
        loss calculation, and visualization of the results.
        """
        # Operational modes and data pipeline
        self.mode = args.mode
        self.load_mode = args.load_mode
        self.data_loader = data_loader
        self.valid_loader = valid_loader # NEW variable

        # Device configuration (GPU vs CPU)
        if args.device:
            self.device = torch.device(args.device)
        else:
            self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        # Normalization and truncation limits for Hounsfield Units (HU)
        self.norm_range_min = args.norm_range_min
        self.norm_range_max = args.norm_range_max
        self.trunc_min = args.trunc_min
        self.trunc_max = args.trunc_max

        # File paths and hardware config
        self.save_path = args.save_path
        self.multi_gpu = args.multi_gpu

        # Training hyperparameters
        self.num_epochs = args.num_epochs
        self.print_iters = args.print_iters
        self.decay_iters = args.decay_iters
        self.save_iters = args.save_iters
        self.test_iters = args.test_iters
        self.result_fig = args.result_fig

        self.patch_size = args.patch_size

        # Masks
        self.ismask = args.ismask

        # RED-CNN model

        # Initialize
        self.REDCNN = RED_CNN()

        # If multiple GPUs are available, wrap the model in DataParallel to split the workload
        if (self.multi_gpu) and (torch.cuda.device_count() > 1):
            print('Use {} GPUs'.format(torch.cuda.device_count()))
            self.REDCNN = nn.DataParallel(self.REDCNN)
        self.REDCNN.to(self.device)

        # ---------------------------------------------------------
        # NEW: Fine-tuning setup (Decoder-only OR Full Network)
        # ---------------------------------------------------------
        if self.mode in ['train_decoder', 'train_fine_tune']:
            # 1. Load the pre-trained weights (using test_iters to specify which checkpoint to load)
            self.load_model(args.test_iters)
            print(f"Loaded weights from iteration {args.test_iters} for {self.mode}.")

            # 2. Freeze the encoder layers ONLY if in decoder-only mode
            if self.mode == 'train_decoder':
                for name, param in self.REDCNN.named_parameters():
                    # The encoder layers are named 'conv1' to 'conv5'
                    # The decoder layers are named 'tconv1' to 'tconv5'
                    if 'conv' in name and 'tconv' not in name:
                        param.requires_grad = False
                        print(f"Froze layer: {name}")
            else:
                print("Fine-tuning entire network. No layers frozen.")
        # ---------------------------------------------------------

        self.criterion = nn.MSELoss()
        # ---old:
        #self.lr = args.lr

        # 3. Filter parameters so the optimizer only updates the unfrozen layers
        # trainable_params = filter(lambda p: p.requires_grad, self.REDCNN.parameters())
        # self.optimizer = optim.Adam(trainable_params, self.lr)
        # self.optimizer = optim.Adam(self.REDCNN.parameters(), self.lr)
        #------
        # ---------------------------------------------------------
        # NEW: Separate LRs and PyTorch Schedulers
        # ---------------------------------------------------------
        enc_params = []
        dec_params = []

        for name, param in self.REDCNN.named_parameters():
            if not param.requires_grad:
                continue # Skip frozen layers
            if 'conv' in name and 'tconv' not in name:
                enc_params.append(param)
            elif 'tconv' in name:
                dec_params.append(param)

        # Use specific LRs if provided, otherwise fall back to base lr
        lr_enc = args.lr_encoder if args.lr_encoder is not None else args.lr
        lr_dec = args.lr_decoder if args.lr_decoder is not None else args.lr

        # Safely build the optimizer groups (prevents crashes if encoder is fully frozen)
        param_groups = []
        if len(enc_params) > 0:
            param_groups.append({'params': enc_params, 'lr': lr_enc})
        if len(dec_params) > 0:
            param_groups.append({'params': dec_params, 'lr': lr_dec})

        self.optimizer = optim.Adam(param_groups)

        # Setup base scheduler
        if args.decay_type == 'step':
            main_scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=args.decay_epochs, gamma=0.5)
        elif args.decay_type == 'cosine':
            main_scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=args.num_epochs - args.warmup_epochs)
        else:
            # Dummy scheduler that keeps LR constant
            main_scheduler = optim.lr_scheduler.MultiStepLR(self.optimizer, milestones=[], gamma=1)

        # Wrap with Linear Warmup if requested
        if args.warmup_epochs > 0:
            warmup_scheduler = optim.lr_scheduler.LinearLR(self.optimizer, start_factor=0.1, end_factor=1.0, total_iters=args.warmup_epochs)
            self.scheduler = optim.lr_scheduler.SequentialLR(self.optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[args.warmup_epochs])
        else:
            self.scheduler = main_scheduler
        # ---------------------------------------------------------


    def save_model(self, iter_):
        """Saves the model's weights (state_dict) at a specific iteration."""
        f = os.path.join(self.save_path, 'REDCNN_{}iter.ckpt'.format(iter_))
        torch.save(self.REDCNN.state_dict(), f)

    def load_model(self, iter_):
        """Loads saved model weights. Handles multi-GPU formatting issues."""
        f = os.path.join(self.save_path, 'REDCNN_{}iter.ckpt'.format(iter_))
        if self.multi_gpu:
            # When a model is saved using nn.DataParallel, PyTorch adds 'module.'
            # to the beginning of every layer name. This loop removes that prefix
            # so the weights can be loaded correctly.
            state_d = OrderedDict()
            for k, v in torch.load(f):
                n = k[7:] # Skips the first 7 characters ('module.')
                state_d[n] = v
            self.REDCNN.load_state_dict(state_d)
        else:
            # Standard single-GPU loading
            self.REDCNN.load_state_dict(torch.load(f))

    def lr_decay(self):
        """Cuts the learning rate in half. Used to fine-tune the model as it converges."""
        lr = self.lr * 0.5
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

    def denormalize_(self, image):
        """Converts image tensors back from [0, 1] range to their original scale (HU)."""
        image = image * (self.norm_range_max - self.norm_range_min) + self.norm_range_min
        return image

    def trunc(self, mat):
        """Clips pixel values to ensure they stay within the predefined physical bounds."""
        # truncation used only in test mode so in one hand, the learning wont be affected
        # and in the other hand, in testing it will clip unexpected values.
        mat[mat <= self.trunc_min] = self.trunc_min
        mat[mat >= self.trunc_max] = self.trunc_max
        return mat

    def save_fig(self, x, y, pred, fig_name, original_result, pred_result):
        """Generates and saves a side-by-side comparison image of the input, target, and output."""
        # Convert PyTorch tensors to NumPy arrays for Matplotlib
        x, y, pred = x.numpy(), y.numpy(), pred.numpy()
        f, ax = plt.subplots(1, 3, figsize=(30, 10))
        # Plot Metal Artifacts (Input)
        ax[0].imshow(x, cmap=plt.cm.gray, vmin=self.trunc_min, vmax=self.trunc_max)
        ax[0].set_title('Metal Artifacts', fontsize=30)
        ax[0].set_xlabel("PSNR: {:.4f}\nSSIM: {:.4f}\nRMSE: {:.4f}".format(original_result[0], original_result[1], original_result[2]), fontsize=20)
        # Plot Prediction (Model Output)
        ax[1].imshow(pred, cmap=plt.cm.gray, vmin=self.trunc_min, vmax=self.trunc_max)
        ax[1].set_title('Result', fontsize=30)
        ax[1].set_xlabel("PSNR: {:.4f}\nSSIM: {:.4f}\nRMSE: {:.4f}".format(pred_result[0], pred_result[1], pred_result[2]), fontsize=20)
        # Plot Clean (Ground Truth Target)
        ax[2].imshow(y, cmap=plt.cm.gray, vmin=self.trunc_min, vmax=self.trunc_max)
        ax[2].set_title('Clean', fontsize=30)

        # In a notebook, we can both save it and display it (Save to disk and display in the notebook output)
        f.savefig(os.path.join(self.save_path, 'fig', 'result_{}.png'.format(fig_name)))
        plt.show()
        plt.close()

    def train(self):
        """The main training loop."""
        train_losses = []
        epoch_train_losses = []
        epoch_lrs = [] # NEW: Track learning rate per epoch
        valid_losses = []
        self.valid_psnr_deltas = []
        self.valid_ssim_deltas = []
        self.valid_rmse_deltas = []
        total_iters = 0
        start_time = time.time()
        for epoch in range(1, self.num_epochs):
            self.REDCNN.train(True)
            # x = low dose input, y = normal dose ground truth
            epoch_loss = 0
            # Note: Unpacking 3 variables here because MAR loader returns img_id
            for iter_, (x, y, _) in enumerate(self.data_loader):
                total_iters += 1

                # Add a channel dimension (unsqueeze) and move data to the GPU/CPU
                x = x.unsqueeze(0).float().to(self.device)
                y = y.unsqueeze(0).float().to(self.device)

                # If processing in patches rather than full images, reshape the tensors
                if self.patch_size:
                    #for Conv2d, reshapes x into a 4D tensor with a single channel and a patch_size × patch_size spatial dimension, while automatically computing batch size.
                    x = x.view(-1, 1, self.patch_size, self.patch_size)
                    y = y.view(-1, 1, self.patch_size, self.patch_size)

                pred = self.REDCNN(x)
                loss = self.criterion(pred, y)
                self.REDCNN.zero_grad()
                self.optimizer.zero_grad()

                loss.backward()
                self.optimizer.step()
                loss_val = loss.item()
                train_losses.append(loss_val)
                epoch_loss += loss_val

                if total_iters % self.print_iters == 0:
                    print("STEP [{}], EPOCH [{}/{}], ITER [{}/{}] \nLOSS: {:.8f}, TIME: {:.1f}s".format(
                        total_iters, epoch, self.num_epochs, iter_+1, len(self.data_loader), loss_val, time.time() - start_time))

                # Decay learning rate at specific intervals
                if total_iters % self.decay_iters == 0:
                    self.lr_decay()

                # Save model checkpoints at specific intervals
                if total_iters % self.save_iters == 0:
                    self.save_model(total_iters)
                    np.save(os.path.join(self.save_path, 'loss_{}_iter.npy'.format(total_iters)), np.array(train_losses))

            # Record average training loss for the epoch
            epoch_train_losses.append(epoch_loss / len(self.data_loader))
            # ---- NEW: Scheduler Step & Metric Saving ----
            # Record the current LR (taking the last parameter group as reference)
            current_lr = self.optimizer.param_groups[-1]['lr']
            epoch_lrs.append(current_lr)

            # Step the scheduler forward one epoch
            self.scheduler.step()

            # Save the epoch-level tracking arrays to disk
            np.save(os.path.join(self.save_path, 'epoch_train_loss.npy'), np.array(epoch_train_losses))
            np.save(os.path.join(self.save_path, 'epoch_lrs.npy'), np.array(epoch_lrs))
            # ---------------------------------------------
            # ---- NEW: Validation Loop with Delta Metrics ----
            if self.valid_loader is not None:
                self.REDCNN.eval()
                val_loss = 0.0
                val_psnr_d, val_ssim_d, val_rmse_d = 0.0, 0.0, 0.0

                with torch.no_grad():
                    # Unpacking 3 variables here as well
                    for val_x, val_y, _ in self.valid_loader:
                        val_x = val_x.unsqueeze(0).float().to(self.device)
                        val_y = val_y.unsqueeze(0).float().to(self.device)

                        if self.patch_size:
                            val_x = val_x.view(-1, 1, self.patch_size, self.patch_size)
                            val_y = val_y.view(-1, 1, self.patch_size, self.patch_size)

                        val_pred = self.REDCNN(val_x)
                        val_loss += self.criterion(val_pred, val_y).item()

                        # Move to CPU, denormalize, and apply HU truncation
                        x_val_cpu = self.trunc(self.denormalize_(val_x.cpu().detach()))
                        y_val_cpu = self.trunc(self.denormalize_(val_y.cpu().detach()))
                        pred_val_cpu = self.trunc(self.denormalize_(val_pred.cpu().detach()))

                        data_range = self.trunc_max - self.trunc_min

                        # Using mask=None for global validation metrics
                        ori_res, pred_res = compute_measure(x_val_cpu, y_val_cpu, pred_val_cpu, data_range, mask=None)

                        val_psnr_d += (pred_res[0] - ori_res[0])
                        val_ssim_d += (pred_res[1] - ori_res[1])
                        val_rmse_d += (pred_res[2] - ori_res[2])

                # Average over all validation batches
                num_batches = len(self.valid_loader)
                valid_losses.append(val_loss / num_batches)

                self.valid_psnr_deltas.append(val_psnr_d / num_batches)
                self.valid_ssim_deltas.append(val_ssim_d / num_batches)
                self.valid_rmse_deltas.append(val_rmse_d / num_batches)

                print("====> EPOCH [{}/{}] Val Loss: {:.6f} | PSNR Delta: {:.4f} | SSIM Delta: {:.4f}".format(
                    epoch, self.num_epochs, (val_loss / num_batches), (val_psnr_d / num_batches), (val_ssim_d / num_batches)))

                # Save arrays to disk
                np.save(os.path.join(self.save_path, 'valid_loss.npy'), np.array(valid_losses))
                np.save(os.path.join(self.save_path, 'valid_psnr_delta.npy'), np.array(self.valid_psnr_deltas))
                np.save(os.path.join(self.save_path, 'valid_ssim_delta.npy'), np.array(self.valid_ssim_deltas))
                np.save(os.path.join(self.save_path, 'valid_rmse_delta.npy'), np.array(self.valid_rmse_deltas))
        # Saving last iteration model
        self.save_model(total_iters)
        print(f"Automatically saved final model at iteration {total_iters}")
        # ---- NEW: Plotting ----
        if self.valid_loader is not None:
            self.plot_losses(epoch_train_losses, valid_losses)

    def plot_losses(self, train_losses, valid_losses):
        """Plots training vs validation loss curve at the end of training."""
        plt.figure(figsize=(10, 5))
        plt.plot(train_losses, label='Train Loss (Epoch Avg)')
        plt.plot(valid_losses, label='Validation Loss')
        plt.xlabel('Epochs')
        plt.ylabel('MSE Loss')
        plt.title('Training and Validation Loss Curve')
        plt.legend()
        plt.grid(True)

        # Save and display
        fig_path = os.path.join(self.save_path, 'fig', 'loss_curve.png')
        plt.savefig(fig_path)
        print(f"Loss curve saved to {fig_path}")
        plt.show()
        plt.close()

    def test(self):
        # Refresh the model to ensure a clean slate, then load the desired saved weights
        del self.REDCNN
        self.REDCNN = RED_CNN().to(self.device)
        self.load_model(self.test_iters)

        #ori_psnr_avg, ori_ssim_avg, ori_rmse_avg = 0, 0, 0
        #pred_psnr_avg, pred_ssim_avg, pred_rmse_avg = 0, 0, 0
        # 1. global scores:
        g_ori_psnr, g_ori_ssim, g_ori_rmse = 0, 0, 0
        g_pred_psnr, g_pred_ssim, g_pred_rmse = 0, 0, 0
        # 2. roi scores:
        a_ori_psnr, a_ori_ssim, a_ori_rmse = 0, 0, 0
        a_pred_psnr, a_pred_ssim, a_pred_rmse = 0, 0, 0

        # path for masks directory
        mask_dir = '/content/DATA_MAR/test/metal_masks/'

        with torch.no_grad():
            for i, (x, y, img_id) in enumerate(self.data_loader):
                shape_ = x.shape[-1]
                x = x.unsqueeze(0).float().to(self.device)
                y = y.unsqueeze(0).float().to(self.device)

                pred = self.REDCNN(x)

                # Move tensors to CPU, remove from graph (.detach()), reshape to 2D, denormalize, and clip values
                x = self.trunc(self.denormalize_(x.view(shape_, shape_).cpu().detach()))
                y = self.trunc(self.denormalize_(y.view(shape_, shape_).cpu().detach()))
                pred = self.trunc(self.denormalize_(pred.view(shape_, shape_).cpu().detach()))

                data_range = self.trunc_max - self.trunc_min

                # Accumulate the metrics to calculate an average later
                # loading mask for current test sample
                curr_img_id = img_id[0]
                mask_filename = f"training_body_metalonlymask_img{curr_img_id}_512x512x1.raw"
                mask_path = os.path.join(mask_dir, mask_filename)

                artifact_mask = None
                if os.path.exists(mask_path) and self.ismask:
                    _, artifact_mask, _ = get_roi_masks(mask_path, shape=(shape_, shape_), dilation_iterations=20)
                else:
                    print(f"\nWarning: Mask not found for ID {curr_img_id} at {mask_path}")

                # 1. global
                g_ori, g_pred = compute_measure(x, y, pred, data_range, mask=None)
                g_ori_psnr += g_ori[0]; g_ori_ssim += g_ori[1]; g_ori_rmse += g_ori[2]
                g_pred_psnr += g_pred[0]; g_pred_ssim += g_pred[1]; g_pred_rmse += g_pred[2]
                # 2. artifact roi
                if artifact_mask is not None:
                    a_ori, a_pred = compute_measure(x, y, pred, data_range, mask=artifact_mask)
                    a_ori_psnr += a_ori[0]; a_ori_ssim += a_ori[1]; a_ori_rmse += a_ori[2]
                    a_pred_psnr += a_pred[0]; a_pred_ssim += a_pred[1]; a_pred_rmse += a_pred[2]

                if self.result_fig:
                    #if a_ori is not None:
                     # self.save_fig(x, y, pred, i, f"{curr_img_id}", a_ori, a_pred, g_ori, g_pred)
                   # else:
                  self.save_fig(x, y, pred, f"{curr_img_id}", g_ori, g_pred)


                printProgressBar(i, len(self.data_loader), prefix="Compute measurements ..", suffix='Complete', length=25)

            num_samples = len(self.data_loader)
            print('\n\n=== Global Results ===')
            print(f'Original -> PSNR: {g_ori_psnr/num_samples:.4f}, SSIM: {g_ori_ssim/num_samples:.4f}, RMSE: {g_ori_rmse/num_samples:.4f}')
            print(f'Predict  -> PSNR: {g_pred_psnr/num_samples:.4f}, SSIM: {g_pred_ssim/num_samples:.4f}, RMSE: {g_pred_rmse/num_samples:.4f}')

            print('\n=== Artifact Zone Results ===')
            print(f'Original -> PSNR: {a_ori_psnr/num_samples:.4f}, SSIM: {a_ori_ssim/num_samples:.4f}, RMSE: {a_ori_rmse/num_samples:.4f}')
            print(f'Predict  -> PSNR: {a_pred_psnr/num_samples:.4f}, SSIM: {a_pred_ssim/num_samples:.4f}, RMSE: {a_pred_rmse/num_samples:.4f}\n')

**Cell 7:** Running Main to train or test the model.

*  Initilize parameters: a path for saving models and figures, data loader for test or train, patch_size, batch_size etc.
*   Initialize the Solver for training or testing
*   Pass parameters as arguments for the solver
*  Execute the Solver according to it's mode, train or test.

**Based on**: main.py

In [ ]:
def main(args):
    # Optimizes CuDNN for faster training if your input sizes don't change
    cudnn.benchmark = True

    # Create directories for saving models and figures if they don't exist
    if not os.path.exists(args.save_path):
        os.makedirs(args.save_path)
        print('Create path : {}'.format(args.save_path))

    if args.result_fig:
        fig_path = os.path.join(args.save_path, 'fig')
        if not os.path.exists(fig_path):
            os.makedirs(fig_path)
            print('Create path : {}'.format(fig_path))

    # Initialize the  train/test data loader
    data_loader = get_loader(mode=args.mode,
                             load_mode=args.load_mode,
                             saved_path=args.saved_path,
                             patch_n=(args.patch_n if args.mode in ['train', 'train_decoder', 'train_fine_tune'] else None),
                             patch_size=(args.patch_size if args.mode in ['train', 'train_decoder', 'train_fine_tune'] else None),
                             transform=args.transform,
                             batch_size=(args.batch_size if args.mode in ['train', 'train_decoder', 'train_fine_tune'] else 1),
                             num_workers=args.num_workers)

    # NEW: Initialize validation data loader if in train mode OR train_decoder mode
    valid_loader = None
    if args.mode in ['train', 'train_decoder', 'train_fine_tune']:
        valid_loader = get_loader(mode='validation',
                                  load_mode=args.load_mode,
                                  saved_path=args.saved_path,
                                  patch_n=args.patch_n,
                                  patch_size=args.patch_size,
                                  transform=args.transform,
                                  batch_size=args.batch_size,
                                  num_workers=args.num_workers)

    # Pass the valid_loader to the Solver
    solver = Solver(args, data_loader, valid_loader)

    # Execute the solver based on mode
    if args.mode in ['train', 'train_decoder', 'train_fine_tune']:
        solver.train()  # train() works for both, as the optimizer already filtered the frozen layers
    elif args.mode == 'test':
        solver.test()


# Optuna Hyperparameter Tuning Pipeline
def objective(trial, args, train_loader, valid_loader):
    # Search Space
    lr = trial.suggest_float("lr", 1e-5, 5e-4, log=True)
    alpha = trial.suggest_float("alpha", 0.6, 0.95)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = RED_CNN_Advanced().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = CombinedLoss(alpha=alpha, data_range=1.0)

    # Fast evaluation loop
    epochs_for_tuning = 5

    for epoch in range(epochs_for_tuning):
        model.train()
        for x, y, _ in train_loader:
            x, y = x.unsqueeze(0).float().to(device), y.unsqueeze(0).float().to(device)
            if args.patch_size:
                x = x.view(-1, 1, args.patch_size, args.patch_size)
                y = y.view(-1, 1, args.patch_size, args.patch_size)

            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

        # Validation for Pruning Check
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y, _ in valid_loader:
                x, y = x.unsqueeze(0).float().to(device), y.unsqueeze(0).float().to(device)
                if args.patch_size:
                    x = x.view(-1, 1, args.patch_size, args.patch_size)
                    y = y.view(-1, 1, args.patch_size, args.patch_size)
                pred = model(x)
                val_loss += criterion(pred, y).item()

        avg_val_loss = val_loss / len(valid_loader)
        trial.report(avg_val_loss, epoch)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return avg_val_loss

def run_optuna(args):
    cudnn.benchmark = True

    train_loader = get_loader(mode='train', load_mode=args.load_mode, saved_path=args.saved_path, patch_n=args.patch_n, patch_size=args.patch_size, batch_size=args.batch_size, num_workers=args.num_workers)
    valid_loader = get_loader(mode='validation', load_mode=args.load_mode, saved_path=args.saved_path, patch_n=args.patch_n, patch_size=args.patch_size, batch_size=args.batch_size, num_workers=args.num_workers)

    pruner = optuna.pruners.MedianPruner(n_startup_trials=2, n_warmup_steps=1)
    study = optuna.create_study(direction="minimize", pruner=pruner)

    print("Starting Optuna optimization. Pruning is ENABLED.")
    study.optimize(lambda trial: objective(trial, args, train_loader, valid_loader), n_trials=10)

    print("\n--- OPTIMIZATION FINISHED ---")
    print("Best hyperparameters: ", study.best_params)

    pruned_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.PRUNED])
    complete_trials = study.get_trials(deepcopy=False, states=[optuna.trial.TrialState.COMPLETE])
    print(f"Trials pruned (Compute Saved!): {len(pruned_trials)}")
    print(f"Trials completed: {len(complete_trials)}")

    return study.best_params


# Execution Block (Conditionals)
if __name__ == "__main__":
    parser = argparse.ArgumentParser()

    # Define operational mode: 'train' | 'train_decoder' | 'train_fine_tune' | 'test' | 'optuna'
    parser.add_argument('--mode', type=str, default='optuna')
    parser.add_argument('--load_mode', type=int, default=0)

    # Paths
    parser.add_argument('--data_path', type=str, default='/content/DATA_MAR/')
    parser.add_argument('--saved_path', type=str, default='/content/npy_img/')
    parser.add_argument('--save_path', type=str, default='/content/save')
    parser.add_argument('--result_fig', type=bool, default=False)

    # Image Normalization limits
    parser.add_argument('--norm_range_min', type=float, default=-1024.0)
    parser.add_argument('--norm_range_max', type=float, default=3072.0)
    parser.add_argument('--trunc_min', type=float, default=-160.0)
    parser.add_argument('--trunc_max', type=float, default=240.0)

    # Patch Config
    parser.add_argument('--transform', type=bool, default=False)
    parser.add_argument('--patch_n', type=int, default=10)
    parser.add_argument('--patch_size', type=int, default=64)
    parser.add_argument('--batch_size', type=int, default=16)

    # Standard Training Hyperparameters
    parser.add_argument('--num_epochs', type=int, default=100)
    parser.add_argument('--print_iters', type=int, default=20)
    parser.add_argument('--decay_iters', type=int, default=3000)
    parser.add_argument('--save_iters', type=int, default=1000)
    parser.add_argument('--test_iters', type=int, default=12000)
    parser.add_argument('--lr', type=float, default=1e-5)
    parser.add_argument('--lr_encoder', type=float, default=None)
    parser.add_argument('--lr_decoder', type=float, default=None)
    parser.add_argument('--decay_type', type=str, default='none', choices=['none', 'step', 'cosine'])
    parser.add_argument('--decay_epochs', type=int, default=30)
    parser.add_argument('--warmup_epochs', type=int, default=0)

    # Hardware Config
    parser.add_argument('--device', type=str)
    parser.add_argument('--multi_gpu', type=bool, default=False)
    parser.add_argument('--num_workers', type=int, default=2)

    # Masks
    parser.add_argument('--ismask', type=int, default=False)

    args = parser.parse_args(args=[])


    # Master Execution Switch
    print(f"Running Colab Pipeline in mode: {args.mode}\n")

    if args.mode == 'optuna':
        best_parameters = run_optuna(args)

    elif args.mode in ['train', 'train_decoder', 'train_fine_tune', 'test']:
        main(args)

    else:
        print(f"Error: Unknown mode '{args.mode}' selected. Please check your arguments.")

[I 2026-07-05 15:32:00,837] A new study created in memory with name: no-name-2ff0716e-d6af-4f66-8388-d7ca02c0a175


Running Colab Pipeline in mode: optuna

Starting Optuna optimization. Pruning is ENABLED.


[I 2026-07-05 15:56:35,143] Trial 0 finished with value: 0.3612039014697075 and parameters: {'lr': 5.669549560558313e-05, 'alpha': 0.7456038979807678}. Best is trial 0 with value: 0.3612039014697075.
[I 2026-07-05 16:19:32,201] Trial 1 finished with value: 0.669126883149147 and parameters: {'lr': 0.00021428901433430013, 'alpha': 0.7603060847914915}. Best is trial 0 with value: 0.3612039014697075.
[I 2026-07-05 16:29:12,607] Trial 2 pruned. 
[I 2026-07-05 16:53:38,800] Trial 3 finished with value: 0.277068305760622 and parameters: {'lr': 3.033765895791955e-05, 'alpha': 0.6298551398496549}. Best is trial 3 with value: 0.277068305760622.
[I 2026-07-05 17:18:02,393] Trial 4 finished with value: 0.4232120253145695 and parameters: {'lr': 0.00034871890290012747, 'alpha': 0.9440243498509633}. Best is trial 3 with value: 0.277068305760622.
[I 2026-07-05 17:27:23,072] Trial 5 pruned. 
[I 2026-07-05 17:37:04,157] Trial 6 pruned. 
[I 2026-07-05 17:46:51,403] Trial 7 pruned. 
[I 2026-07-05 17:56:04


--- OPTIMIZATION FINISHED ---
Best hyperparameters:  {'lr': 3.033765895791955e-05, 'alpha': 0.6298551398496549}
Trials pruned (Compute Saved!): 6
Trials completed: 4
